[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/measurement-day.ipynb)

# Measurement day · every Tier-1 number in one run

**Hardware:** any Colab **TPU runtime** (Runtime → Change runtime type → v5e-1 or v6e-1). Run the first cell; if it installs anything, Runtime → Restart session, then Run all.

One pass produces the measured values for gates 00 through 03, the causal-scaling number, the bf16 differential results, and the two museum error captures only a real TPU compile can produce. The final cell prints one JSON blob: paste it back and the site's gates and bench update from it. Kernels here are inlined verbatim from labs 1.2, 1.3, 3.2, and 3.4 so this notebook stands alone.


In [ ]:
# Colab TPU images sometimes pair jax with an older libtpu; every
# pallas_call then dies with "Failed to deserialize the Mosaic module:
# Unsupported version". Installing jax[tpu] at the preinstalled jax
# version pulls the libtpu that matches it. If pip reports installs or
# upgrades below: Runtime -> Restart session, then Run all.
import importlib.metadata as md
v = md.version("jax")
print(f"jax {v}: syncing libtpu to match")
!pip install -q "jax[tpu]==$v"


In [ ]:
import time, json, functools
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
CHIP = jax.devices()[0].device_kind if ON_TPU else "none"
INTERP = not ON_TPU
if not ON_TPU:
    print("NOT a TPU runtime: Runtime -> Change runtime type -> a TPU (v5e-1 or v6e-1), then Run all again.")
RESULTS = {"chip": CHIP, "results": {}}

def bench(fn, *args, reps=20):
    fn(*args).block_until_ready()
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args).block_until_ready()
        ts.append(time.perf_counter() - t0)
    return float(np.median(ts)) * 1e6  # us

def record(key, value):
    RESULTS["results"][key] = value
    print(f"  -> {key} = {value}")


## Gate 00 · rooflines, predicted then measured

In [ ]:
# gate 00: the five roofline predictions, measured
CHIPS = {"v5e": {"flops": 1.97e14, "hbm_bw": 8.2e11},
         "v6e": {"flops": 9.2e14, "hbm_bw": 1.6e12}}
CHIP_KEY = "v6e" if "v6" in CHIP.lower() else "v5e"
def predict(flops, bytes_moved):
    c = CHIPS[CHIP_KEY]
    return max(flops / c["flops"], bytes_moved / c["hbm_bw"]) * 1e6

N = 4096
preds = {
    "matmul NxNxN":        predict(2 * N**3, 2 * 3 * N**2),
    "matmul skinny 8xNxN": predict(2 * 8 * N**2, 2 * (8*N + N*N + 8*N)),
    "elementwise add":     predict(N**2, 2 * 3 * N**2),
    "softmax rows":        predict(5 * N**2, 2 * 2 * N**2),
    "reduce_sum":          predict(N**2, 2 * (N**2 + N)),
}
if ON_TPU:
    key = jax.random.key(0)
    a = jax.random.normal(key, (N, N), jnp.bfloat16)
    b = jax.random.normal(key, (N, N), jnp.bfloat16)
    s8 = jax.random.normal(key, (8, N), jnp.bfloat16)
    cases = {
        "matmul NxNxN": (jax.jit(lambda x, y: x @ y), a, b),
        "matmul skinny 8xNxN": (jax.jit(lambda x, y: x @ y), s8, b),
        "elementwise add": (jax.jit(lambda x, y: x + y), a, b),
        "softmax rows": (jax.jit(lambda x: jax.nn.softmax(x, axis=-1)), a),
        "reduce_sum": (jax.jit(lambda x: jnp.sum(x, axis=-1)), a),
    }
    worst = 0.0
    for name, (fn, *args) in cases.items():
        us = bench(fn, *args)
        ratio = us / preds[name]
        worst = max(worst, ratio, 1 / ratio if ratio < 1 else ratio)
        record(f"lab0.1/{name}", {"measured_us": round(us, 1), "predicted_us": round(preds[name], 1), "ratio": round(ratio, 2)})
    record("gate00/chip_constants", CHIP_KEY)
    record("gate00/worst_ratio", round(worst, 2))

## Gate 01 · the matmul sweep and the fused softmax

In [ ]:
# gate 01a: pallas tiled matmul vs XLA at 4096^3 bf16
def matmul_kernel(a_ref, b_ref, o_ref):
    k = pl.program_id(2)
    @pl.when(k == 0)
    def _():
        o_ref[...] = jnp.zeros_like(o_ref)
    o_ref[...] += jnp.dot(a_ref[...], b_ref[...], preferred_element_type=jnp.float32).astype(o_ref.dtype)

def matmul(a, b, bm, bn, bk):
    m, k = a.shape
    _, n = b.shape
    return pl.pallas_call(
        matmul_kernel,
        grid=(m // bm, n // bn, k // bk),
        in_specs=[pl.BlockSpec((bm, bk), lambda i, j, kk: (i, kk)),
                  pl.BlockSpec((bk, bn), lambda i, j, kk: (kk, j))],
        out_specs=pl.BlockSpec((bm, bn), lambda i, j, kk: (i, j)),
        out_shape=jax.ShapeDtypeStruct((m, n), a.dtype),
        interpret=INTERP,
    )(a, b)

if ON_TPU:
    N = 4096
    a = jax.random.normal(jax.random.key(0), (N, N), jnp.bfloat16)
    b = jax.random.normal(jax.random.key(1), (N, N), jnp.bfloat16)
    xla_us = bench(jax.jit(lambda x, y: x @ y), a, b)
    best = None
    for bm, bn, bk in [(256, 256, 256), (512, 512, 512), (512, 1024, 512), (1024, 512, 512), (512, 512, 1024)]:
        try:
            us = bench(jax.jit(functools.partial(matmul, bm=bm, bn=bn, bk=bk)), a, b)
            print(f"  ({bm},{bn},{bk}): {us:8.0f} us  ratio {us / xla_us:5.2f}x")
            if best is None or us < best[1]:
                best = ((bm, bn, bk), us)
        except Exception as e:
            print(f"  ({bm},{bn},{bk}): failed: {str(e)[:80]}")
    record("gate01/matmul", {"xla_us": round(xla_us, 1), "pallas_best_us": round(best[1], 1),
                             "best_block": list(best[0]), "ratio": round(best[1] / xla_us, 3)})

In [ ]:
# gate 01b: fused softmax vs the unfused chain at 32k rows
def softmax_kernel(x_ref, o_ref):
    x = x_ref[...].astype(jnp.float32)
    m = jnp.max(x, axis=-1, keepdims=True)
    e = jnp.exp(x - m)
    o_ref[...] = (e / jnp.sum(e, axis=-1, keepdims=True)).astype(o_ref.dtype)

def softmax(x, rows=64):
    n, d = x.shape
    return pl.pallas_call(
        softmax_kernel,
        grid=(n // rows,),
        in_specs=[pl.BlockSpec((rows, d), lambda i: (i, 0))],
        out_specs=pl.BlockSpec((rows, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        interpret=INTERP,
    )(x)

if ON_TPU:
    big = jax.random.normal(jax.random.key(0), (32768, 512), jnp.bfloat16)
    m_fn = jax.jit(lambda v: jnp.max(v, -1, keepdims=True))
    e_fn = jax.jit(jnp.exp)
    s_fn = jax.jit(lambda v: jnp.sum(v, -1, keepdims=True))
    def unfused(x):
        m = m_fn(x); e = e_fn(x - m); return e / s_fn(e)
    unf = bench(unfused, big)
    xla = bench(jax.jit(lambda v: jax.nn.softmax(v, -1)), big)
    pal = bench(jax.jit(softmax), big)
    record("gate01/softmax", {"unfused_us": round(unf, 1), "xla_fused_us": round(xla, 1),
                              "pallas_us": round(pal, 1), "beats_unfused": bool(pal < unf)})

## Gate 02 · the spill in the compiler's own output

In [ ]:
# gate 02: the spill, hunted in XLA's own output at seq 8192
def naive_attention(q, k, v):
    s = q @ k.T
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    l = jnp.sum(p, axis=-1, keepdims=True)
    return (p / l) @ v

if ON_TPU:
    S, D = 8192, 128
    args = [jax.ShapeDtypeStruct((S, D), jnp.bfloat16)] * 3
    hlo = jax.jit(naive_attention).lower(*args).compile().as_text()
    spills = [ln for ln in hlo.splitlines() if "bf16[8192,8192]" in ln and "fusion" in ln]
    xs = [jax.random.normal(jax.random.key(i), (S, D), jnp.bfloat16) for i in range(3)]
    naive_us = bench(jax.jit(naive_attention), *xs)
    record("gate02/spill", {"score_matrix_mb": round(S * S * 2 / 1e6),
                            "fusion_lines_with_full_matrix": len(spills),
                            "naive_us_seq8192": round(naive_us, 1)})

## Gate 03 · flash forward, gradients, and the causal scaling

In [ ]:
# gate 03: flash forward vs references at seq 8192, plus bf16 differential tests
def flash_kernel(q_ref, k_ref, v_ref, o_ref, *, block_kv):
    q = q_ref[...].astype(jnp.float32)
    n_kv = k_ref.shape[0]
    def step(j, state):
        m, l, acc = state
        kb = jax.lax.dynamic_slice_in_dim(k_ref[...], j * block_kv, block_kv).astype(jnp.float32)
        vb = jax.lax.dynamic_slice_in_dim(v_ref[...], j * block_kv, block_kv).astype(jnp.float32)
        s = q @ kb.T
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new[:, None])
        return m_new, l * alpha + jnp.sum(p, axis=-1), acc * alpha[:, None] + p @ vb
    m0 = jnp.full((q.shape[0],), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((q.shape[0],), jnp.float32)
    acc0 = jnp.zeros((q.shape[0], v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_kv // block_kv, step, (m0, l0, acc0))
    o_ref[...] = (acc / l[:, None]).astype(o_ref.dtype)

def flash(q, k, v, block_q=256, block_kv=512):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(flash_kernel, block_kv=block_kv),
        grid=(sq // block_q,),
        in_specs=[pl.BlockSpec((block_q, d), lambda i: (i, 0)),
                  pl.BlockSpec(k.shape, lambda i: (0, 0)),
                  pl.BlockSpec(v.shape, lambda i: (0, 0))],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        interpret=INTERP,
    )(q, k, v)

if ON_TPU:
    S, D = 8192, 128
    q = jax.random.normal(jax.random.key(0), (S, D), jnp.bfloat16)
    k = jax.random.normal(jax.random.key(1), (S, D), jnp.bfloat16)
    v = jax.random.normal(jax.random.key(2), (S, D), jnp.bfloat16)

    ref = jax.jit(naive_attention)(q, k, v)
    got = jax.jit(flash)(q, k, v)
    fwd_err = float(jnp.abs(got.astype(jnp.float32) - ref.astype(jnp.float32)).max())

    flash_us = bench(jax.jit(flash), q, k, v)
    try:
        ref_fast = jax.jit(lambda q, k, v: jax.nn.dot_product_attention(q[None, :, None], k[None, :, None], v[None, :, None])[0, :, 0])
        ref_us = bench(ref_fast, q, k, v)
    except Exception:
        ref_us = None
    naive_us = bench(jax.jit(naive_attention), q, k, v)
    record("gate03/flash", {"fwd_max_err_bf16": fwd_err, "flash_us": round(flash_us, 1),
                            "naive_us": round(naive_us, 1),
                            "reference_us": round(ref_us, 1) if ref_us else None,
                            "vs_naive": round(naive_us / flash_us, 2),
                            "vs_reference": round(flash_us / ref_us, 2) if ref_us else None})

In [ ]:
# gate 03b: custom_vjp gradients differential-tested in bf16 on chip
def attention_ref(q, k, v):
    return jax.nn.softmax((q @ k.T).astype(jnp.float32), axis=-1).astype(q.dtype) @ v

@jax.custom_vjp
def attention_cv(q, k, v):
    return attention_ref(q, k, v)

def fwd(q, k, v):
    s = (q @ k.T).astype(jnp.float32)
    m = jnp.max(s, axis=-1, keepdims=True)
    p = jnp.exp(s - m)
    l = jnp.sum(p, axis=-1, keepdims=True)
    o = ((p / l).astype(q.dtype)) @ v
    return o, (q, k, v, o, m, l)

def bwd(res, do):
    q, k, v, o, m, l = res
    p = (jnp.exp((q @ k.T).astype(jnp.float32) - m) / l).astype(q.dtype)
    dv = p.T @ do
    d = jnp.sum((do * o).astype(jnp.float32), axis=-1, keepdims=True).astype(q.dtype)
    dp = do @ v.T
    ds = p * (dp - d)
    return ds @ k, ds.T @ q, dv

attention_cv.defvjp(fwd, bwd)

if ON_TPU:
    q = jax.random.normal(jax.random.key(0), (512, 128), jnp.bfloat16)
    k = jax.random.normal(jax.random.key(1), (1024, 128), jnp.bfloat16)
    v = jax.random.normal(jax.random.key(2), (1024, 128), jnp.bfloat16)
    g1 = jax.grad(lambda *a: jnp.sum(jnp.tanh(attention_cv(*a).astype(jnp.float32))), argnums=(0, 1, 2))(q, k, v)
    g2 = jax.grad(lambda *a: jnp.sum(jnp.tanh(attention_ref(*a).astype(jnp.float32))), argnums=(0, 1, 2))(q, k, v)
    errs = {n: float(jnp.abs(x.astype(jnp.float32) - y.astype(jnp.float32)).max()) for n, x, y in zip("qkv", g1, g2)}
    record("gate03/grads_bf16", {k2: round(v2, 5) for k2, v2 in errs.items()})

In [ ]:
# causal scaling: blocks skipped vs dense at seq 8192
def causal_kernel(q_ref, k_ref, v_ref, o_ref, *, block_q, block_kv):
    qi = pl.program_id(0)
    q = q_ref[...].astype(jnp.float32)
    row0 = qi * block_q
    def step(j, state):
        m, l, acc = state
        kb = jax.lax.dynamic_slice_in_dim(k_ref[...], j * block_kv, block_kv).astype(jnp.float32)
        vb = jax.lax.dynamic_slice_in_dim(v_ref[...], j * block_kv, block_kv).astype(jnp.float32)
        s = q @ kb.T
        cols = j * block_kv + jax.lax.broadcasted_iota(jnp.int32, s.shape, 1)
        rows = row0 + jax.lax.broadcasted_iota(jnp.int32, s.shape, 0)
        s = jnp.where(cols <= rows, s, -jnp.inf)
        m_new = jnp.maximum(m, jnp.max(s, axis=-1))
        alpha = jnp.exp(m - m_new)
        p = jnp.exp(s - m_new[:, None])
        return m_new, l * alpha + jnp.sum(p, axis=-1), acc * alpha[:, None] + p @ vb
    n_live = (row0 + block_q + block_kv - 1) // block_kv
    m0 = jnp.full((block_q,), -jnp.inf, jnp.float32)
    l0 = jnp.zeros((block_q,), jnp.float32)
    acc0 = jnp.zeros((block_q, v_ref.shape[1]), jnp.float32)
    m, l, acc = jax.lax.fori_loop(0, n_live, step, (m0, l0, acc0))
    o_ref[...] = (acc / l[:, None]).astype(o_ref.dtype)

def causal_flash(q, k, v, block_q=256, block_kv=512):
    sq, d = q.shape
    return pl.pallas_call(
        functools.partial(causal_kernel, block_q=block_q, block_kv=block_kv),
        grid=(sq // block_q,),
        in_specs=[pl.BlockSpec((block_q, d), lambda i: (i, 0)),
                  pl.BlockSpec(k.shape, lambda i: (0, 0)),
                  pl.BlockSpec(v.shape, lambda i: (0, 0))],
        out_specs=pl.BlockSpec((block_q, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(q.shape, q.dtype),
        interpret=INTERP,
    )(q, k, v)

if ON_TPU:
    S, D = 8192, 128
    q = jax.random.normal(jax.random.key(0), (S, D), jnp.bfloat16)
    k = jax.random.normal(jax.random.key(1), (S, D), jnp.bfloat16)
    v = jax.random.normal(jax.random.key(2), (S, D), jnp.bfloat16)
    dense_us = bench(jax.jit(flash), q, k, v)
    causal_us = bench(jax.jit(causal_flash), q, k, v)
    record("lab3.4/causal_scaling", {"dense_us": round(dense_us, 1), "causal_us": round(causal_us, 1),
                                     "speedup": round(dense_us / causal_us, 2)})

## The museum's missing exhibits · real Mosaic errors

In [ ]:
# two museum exhibits only a TPU can produce: real Mosaic error text
if ON_TPU:
    captures = {}
    def try_capture(name, fn):
        try:
            fn()
            captures[name] = "DID NOT FAIL"
        except Exception as e:
            captures[name] = str(e)[:1200]
        print(f"  {name}: {captures[name][:110]}")

    def vmem_overflow():
        big = 4096
        x = jnp.ones((big, big), jnp.float32)
        pl.pallas_call(
            lambda x_ref, o_ref: o_ref.__setitem__(..., x_ref[...] * 2),
            in_specs=[pl.BlockSpec((big, big), lambda: (0, 0))],
            out_specs=pl.BlockSpec((big, big), lambda: (0, 0)),
            out_shape=jax.ShapeDtypeStruct((big, big), jnp.float32),
        )(x).block_until_ready()

    def lattice_violation():
        x = jnp.ones((70, 300), jnp.float32)
        pl.pallas_call(
            lambda x_ref, o_ref: o_ref.__setitem__(..., x_ref[...] * 2),
            grid=(10, 3),
            in_specs=[pl.BlockSpec((7, 100), lambda i, j: (i, j))],
            out_specs=pl.BlockSpec((7, 100), lambda i, j: (i, j)),
            out_shape=jax.ShapeDtypeStruct((70, 300), jnp.float32),
        )(x).block_until_ready()

    try_capture("vmem_overflow", vmem_overflow)
    try_capture("lattice_violation", lattice_violation)
    RESULTS["museum_captures"] = captures

## The blob

In [ ]:
print("=" * 60)
print("MEASUREMENT DAY RESULTS · paste this whole blob back")
print("=" * 60)
print(json.dumps(RESULTS, indent=1))